In [4]:
import pandas as pd

from pathlib import Path

import numpy as np
import os
import json

h_to_eV = 27.2114

In [5]:
class PropertiesCharges:
    """Properties from JSON file from Orca NMR calculations"""

    def __init__(self, path):
        """this class reads an orca json file from a given path and extracts properties
        Args:
            path (Path): Path to the orca property.json file
        The class is initialized by reading the json file
        """
        
        self.path = path

        self.results = {}

        if 'singlepoint.json' in os.listdir(path):
            self.singlepointjson = json.load(open(path + 'singlepoint.json'))
        else:
            print("No singlepoint.json file found in the specified path. Results dictionary will be empty.")
            self.singlepointjson = {}

        if 'singlepoint.property.json' in os.listdir(path):
            self.singlepointpropertyjson = json.load(open(path + 'singlepoint.property.json'))
        else:
            print("No singlepoint.property.json file found in the specified path. Results dictionary will be empty.")
            self.singlepointpropertyjson = {}

        if 'optimization.property.json' in os.listdir(path):
            self.optimizationpropertyjson = json.load(open(path + 'optimization.property.json'))
        else:
            print("No optimization.property.json file found in the specified path. Results dictionary will be empty.")
            self.optimizationpropertyjson = {}

        # initialize lists for atoms, bonds, angles, dihedrals
        self.atom_list = None

    def get_scfenergy(self):
        """
        Extract SCF energy and store in results dictionary
        """
        self.results["SCF_energy"] = self.singlepointpropertyjson["Geometries"][-1]['DFT_Energy']['finalEn']

    def get_homolumo(self):
        """
        Extract HOMO and LUMO energies and store in results dictionary
        """

        # MO handling (HOMO/LUMO)
        mos = self.singlepointjson["Molecule"]["MolecularOrbitals"]["MOs"]
        mo_energies = []
        mo_occupancy = []
        for mo in mos:
            mo_energies.append(mo["OrbitalEnergy"])
            mo_occupancy.append(mo["Occupancy"])
        mo_energies = np.array(mo_energies)
        mo_occupancy = np.array(mo_occupancy)

        HOMO_energy = np.max(mo_energies[(mo_occupancy == 1.0) | (mo_occupancy == 2.0)])
        HOMO_energy = HOMO_energy * h_to_eV
        LUMO_energy = np.min(mo_energies[mo_occupancy == 0.0])
        LUMO_energy = LUMO_energy * h_to_eV

        self.results["HOMO_energy"] = HOMO_energy
        self.results["LUMO_energy"] = LUMO_energy

    def get_enthalpy(self):
        """
        Extract enthalpy and store in results dictionary
        """
        self.results["Enthalpy"] = self.optimizationpropertyjson["Geometries"][-1]["THERMOCHEMISTRY_Energies"]["Enthalpy"]

In [6]:
P = PropertiesCharges(path='orca_example/')
P.get_homolumo()
P.get_homolumo()
P.results

No optimization.property.json file found in the specified path. Results dictionary will be empty.


{'HOMO_energy': np.float64(-11.842291455378755),
 'LUMO_energy': np.float64(1.7364257585818368)}